In [1]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from tqdm import tqdm
from collections import Counter

In [2]:
print(torch.cuda.is_available())

True


In [5]:
reviews_df = pd.read_csv("sample_data/final_yelp_dataset.csv")
reviews_df.head(5)

reviews_sample = reviews_df.sample(n = 10000, random_state = 42)

In [6]:
device = 0 if torch.cuda.is_available() else -1
print("Using device:", "GPU" if device == 0 else "CPU")

roberta_model = pipeline(
    "sentiment-analysis",
    model = "cardiffnlp/twitter-roberta-base-sentiment",
    truncation = True,
    max_length = 512,
    device = device
)

Using device: GPU


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [12]:
def get_roberta_sentiments(texts, batch_size=128):
    results = []
    texts = texts.tolist()

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i + batch_size]
        outputs = roberta_model(batch, truncation=True)
        results.extend(outputs)

    return results

In [13]:
print(reviews_sample['text'].str.len().describe())

count    35124.000000
mean       473.038748
std        440.940328
min         31.000000
25%        191.000000
50%        334.000000
75%        599.000000
max       4999.000000
Name: text, dtype: float64


In [14]:
sampled_df = (
    reviews_df
    .groupby(['cuisine', 'tier'], group_keys=False)
    .apply(lambda x: x.sample(min(len(x), 5000), random_state=42))
    .reset_index(drop=True)
)

pivot_table = sampled_df.pivot_table(
    index='cuisine',
    columns='tier',
    aggfunc='size',
    fill_value=0
)
pivot_table = pivot_table[['$', '$$', '$$$']]

print(pivot_table)
print(f"\nTotal reviews: {len(sampled_df)}")

tier         $    $$   $$$
cuisine                   
American  3969  5000  2183
Chinese    661  2520     1
French     103   234   230
Greek      161   744     0
Indian      46  1090     4
Italian    494  5000   483
Japanese    93  4073   173
Korean      73   255     0
Mexican   2011  4150   141
Thai        68   862     0

Total reviews: 35124


/tmp/ipykernel_4935/2015098282.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), 5000), random_state=42))


In [15]:
reviews_sample = sampled_df.copy()
roberta_results = get_roberta_sentiments(reviews_sample['text'])

reviews_sample['roberta_label'] = [r['label'] for r in roberta_results]
reviews_sample['roberta_score'] = [r['score'] for r in roberta_results]

reviews_sample['roberta_sentiment'] = reviews_sample.apply(
    lambda x: x['roberta_score'] if x['roberta_label'] == 'LABEL_2'
    else (0.5 if x['roberta_label'] == 'LABEL_1'
    else 1 - x['roberta_score']),
    axis=1
)

100%|██████████| 275/275 [07:22<00:00,  1.61s/it]


In [16]:
def convert_score(r):
    if r['label'] == "positive":
        return r['score']
    elif r['label'] == "neutral":
        return 0.5
    else:
        return 1 - r['score']
reviews_sample['roberta_score'] = [convert_score(r) for r in roberta_results]


In [17]:
labels = [r['label'] for r in roberta_results]
print(Counter(labels))

Counter({'LABEL_2': 25438, 'LABEL_0': 8501, 'LABEL_1': 1185})


In [18]:
print(reviews_sample[['roberta_score']].describe())

       roberta_score
count   35124.000000
mean        0.107944
std         0.161447
min         0.006052
25%         0.010797
50%         0.025523
75%         0.123417
max         0.663383
